In [11]:

#This code works on RT Struct Masks
# Import the Google Drive helper module from Colab
from google.colab import drive

# Mount your Google Drive so Colab can read/write files there
drive.mount('/content/drive')



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
# Install the required Python packages quietly (-q = less output)
!pip install -q pydicom numpy pandas scipy nibabel



In [19]:
# Root folder that contains all case folders
#ROOT_DIR = "/content/drive/MyDrive/SUV-IBSI/Task 3_suv_computation-main/DRO"
ROOT_DIR = r"/content/drive/MyDrive/SUV-IBSI/Task 3_suv_computation-main/Real Patients"
     #This version assumes each case looks like this:
            # case_001/
                      # PT/
                           # ... PET DICOM files ...
                      # mask/
                           # lesion_mask.nii.gz

# CSV file where SUV summary results will be written
OUTPUT_CSV = r"/content/drive/MyDrive/SUV-IBSI/Task 3_suv_computation-main/Results/Result 5.csv"


# Folder where new SUV-corrected PET DICOM images will be saved
SUV_OUTPUT_DIR = r"/content/drive/MyDrive/SUV-IBSI/Task 3_suv_computation-main/Corrected Images"



In [14]:
# Import tools for file and folder operations
import os

# Import math functions such as log and exp
import math

# Import CSV writer utilities
import csv

# Import copy so we can duplicate DICOM datasets safely before editing
import copy

# Import date and time handling utilities
from datetime import datetime, timedelta
import warnings
from types import SimpleNamespace

# Import NumPy for arrays and numerical operations
import numpy as np

# Import pydicom for reading and writing DICOM medical image files
import pydicom

# Import pandas for tables and DataFrames
import pandas as pd

# Import binary erosion to shrink masks by a chosen number of voxels
from scipy.ndimage import binary_erosion

# Import nibabel to read NIfTI mask files (.nii.gz)
import nibabel as nib

# Import UID generator for creating new DICOM identifiers
from pydicom.uid import generate_uid


# Safely convert a value to float
def safe_float(x, default=None):
    try:
        # If the value is missing or empty, return the default
        if x is None or x == "":
            return default
        # Try converting to float
        return float(x)
    except Exception:
        # If conversion fails, return the default
        return default


# Parse a DICOM TM (time) string
def parse_dicom_tm(tm_str):
    # Return None if empty
    if not tm_str:
        return None

    # Convert to string and remove surrounding spaces
    s = str(tm_str).strip()

    # Default fractional seconds = 0
    frac = 0

    # If there is a decimal point, extract fractional seconds
    if "." in s:
        _, frac_str = s.split(".", 1)
        # Pad or trim to 6 digits for microseconds
        frac = int((frac_str + "000000")[:6])

    # Take the part before decimal and pad to HHMMSS
    base = s.split(".")[0].ljust(6, "0")

    try:
        # Parse hours
        h = int(base[0:2])

        # Parse minutes
        m = int(base[2:4])

        # Parse seconds
        sec = int(base[4:6])

        # Return time components plus microseconds
        return h, m, sec, frac
    except Exception:
        # Return None if parsing fails
        return None


# Parse a DICOM DA (date) string
def parse_dicom_da(da_str):
    # Return None if empty
    if not da_str:
        return None
    try:
        # Parse date in YYYYMMDD format
        return datetime.strptime(str(da_str), "%Y%m%d")
    except Exception:
        # Return None if invalid
        return None


# Combine separate DICOM date and time strings into one datetime
def combine_da_tm(date_str, time_str):
    # Parse the date
    d = parse_dicom_da(date_str)

    # Parse the time
    t = parse_dicom_tm(time_str)

    # If either failed, return None
    if d is None or t is None:
        return None

    # Replace time part of the date object and return full datetime
    return d.replace(hour=t[0], minute=t[1], second=t[2], microsecond=t[3])


# Parse a DICOM DT (datetime) string
def parse_dicom_dt(dt_str):
    # Return None if empty
    if not dt_str:
        return None

    # Convert to clean string
    s = str(dt_str).strip()

    # Remove timezone offset if present, because strptime below does not use it
    for sign in ["+", "-"]:
        if len(s) > 14 and sign in s[14:]:
            s = s.split(sign, 1)[0]
            break

    # Try several possible DICOM datetime formats
    for fmt in ["%Y%m%d%H%M%S.%f", "%Y%m%d%H%M%S", "%Y%m%d%H%M", "%Y%m%d"]:
        try:
            return datetime.strptime(s, fmt)
        except Exception:
            pass

    # If none worked, return None
    return None


# Read a private DICOM tag as datetime if present
def parse_private_dt(ds, tag):
    elem = ds.get(tag)
    if elem is None:
        return None
    value = getattr(elem, "value", None)
    if isinstance(value, bytes):
        value = value.decode("ascii", errors="ignore").strip("\x00 ")
    return parse_dicom_dt(value)


# Find all case folders inside the root directory
def find_case_dirs(root_dir):
    return [
        # Build full path for each folder name
        os.path.join(root_dir, name)
        # Loop over sorted names in the root directory
        for name in sorted(os.listdir(root_dir))
        # Keep only directories
        if os.path.isdir(os.path.join(root_dir, name))
    ]


# Find all PET DICOM files in a PT folder
def find_dicom_files(pt_dir):
    # Start with an empty list
    dicoms = []

    # Walk through all subfolders and files
    for root, _, files in os.walk(pt_dir):
        for f in files:
            # Full file path
            path = os.path.join(root, f)
            try:
                # Read only metadata first for speed
                ds = pydicom.dcmread(path, stop_before_pixels=True, force=True)

                # Keep only PET modality files
                if getattr(ds, "Modality", None) == "PT":
                    dicoms.append(path)
            except Exception:
                # Ignore unreadable files
                pass

    # Fail if no PET files found
    if not dicoms:
        raise FileNotFoundError(f"No PET DICOM files found in {pt_dir}")

    # Return list of PET DICOM file paths
    return dicoms


# Find one NIfTI mask file in a mask folder
def find_mask_file(mask_dir):
    # Allowed mask extensions
    candidates = []

    # Search through folder tree
    for root, _, files in os.walk(mask_dir):
        for f in files:
            lower = f.lower()

            # Keep .nii and .nii.gz files
            if lower.endswith(".nii") or lower.endswith(".nii.gz"):
                candidates.append(os.path.join(root, f))

    # Fail if none found
    if not candidates:
        raise FileNotFoundError(f"No NIfTI mask file found in {mask_dir}")

    # Sort for deterministic behavior
    candidates.sort()

    # Return the first mask file
    return candidates[0]


# Sort PET slices by physical position, not acquisition time.
def sort_pet_slices(datasets):
    def key(ds):
        orientation = getattr(ds, "ImageOrientationPatient", None)
        position = getattr(ds, "ImagePositionPatient", None)
        if orientation is None or len(orientation) != 6 or position is None or len(position) != 3:
            raise ValueError("Missing DICOM geometry; PET slice order cannot be verified")
        row_dir = np.asarray(orientation[:3], dtype=float)
        col_dir = np.asarray(orientation[3:], dtype=float)
        normal = np.cross(row_dir, col_dir)
        return (float(np.dot(np.asarray(position, dtype=float), normal)),
                safe_float(getattr(ds, "InstanceNumber", None), 0))

    if datasets:
        first_orientation = np.asarray(datasets[0].ImageOrientationPatient, dtype=float)
        if any(not np.allclose(np.asarray(ds.ImageOrientationPatient, dtype=float),
                                   first_orientation, atol=1e-5)
               for ds in datasets):
            raise ValueError("Inconsistent PET slice orientation")
    return sorted(datasets, key=key)


# Check that important metadata are consistent across the series
def assert_series_consistency(datasets):
    # Attributes that should not vary slice to slice
    attrs_to_check = [
        "Units",
        "DecayCorrection",
        "PatientWeight",
        "PatientSize",
        "PatientSex",
    ]

    # Check each attribute
    for attr in attrs_to_check:
        # Collect values as strings
        vals = [str(getattr(ds, attr, None)) for ds in datasets]

        # If more than one unique value exists, fail
        if len(set(vals)) > 1:
            raise ValueError(f"Inconsistent series attribute: {attr} -> {set(vals)}")


# Load the PET DICOM series into a 3D volume
def load_pet_series(pt_dir):
    # Find PET files
    dicom_files = find_dicom_files(pt_dir)

    # Read all PET files fully
    datasets = [pydicom.dcmread(p, force=True) for p in dicom_files]

    # Sort slices
    datasets = sort_pet_slices(datasets)

    # Validate consistent metadata
    assert_series_consistency(datasets)

    # Prepare containers
    arrays, slopes, intercepts = [], [], []

    # Loop through each slice
    for ds in datasets:
        # Read pixel array as float64
        pixel = ds.pixel_array.astype(np.float64)

        # Reject enhanced multiframe data
        if pixel.ndim != 2:
            raise NotImplementedError("Enhanced multiframe PET is not supported.")

        # Store raw pixel array
        arrays.append(pixel)

        # Store per-slice rescale slope
        slopes.append(safe_float(getattr(ds, "RescaleSlope", 1.0), 1.0))

        # Store per-slice rescale intercept
        intercepts.append(safe_float(getattr(ds, "RescaleIntercept", 0.0), 0.0))

    # Stack slices into 3D volume [z, y, x]
    volume = np.stack(arrays, axis=0)

    # Return metadata, raw volume, slopes, intercepts
    return (
        datasets,
        volume,
        np.asarray(slopes, dtype=np.float64),
        np.asarray(intercepts, dtype=np.float64),
    )


# Apply DICOM rescale equation to each slice
def apply_rescale(volume, slopes, intercepts):
    # Create empty output array
    out = np.empty_like(volume, dtype=np.float64)

    # Process each slice
    for z in range(volume.shape[0]):
        # Apply slope and intercept
        out[z] = volume[z] * slopes[z] + intercepts[z]

    # Return rescaled values
    return out


# Extract radiopharmaceutical info block from DICOM
def get_radiopharm_item(ds):
    # Get RadiopharmaceuticalInformationSequence
    seq = getattr(ds, "RadiopharmaceuticalInformationSequence", None)

    # Fail if missing
    if not seq or len(seq) == 0:
        raise ValueError("Missing RadiopharmaceuticalInformationSequence")

    # Use the first item
    return seq[0]


# Get patient weight in kg
def get_weight_kg(ds):
    # Read patient weight
    w = safe_float(getattr(ds, "PatientWeight", None))

    # Fail if missing or invalid
    if w is None or w <= 0:
        raise ValueError("Missing or zero PatientWeight")

    # Convert grams to kg if it looks too large
    if w >= 1000:
        w /= 1000.0

    # Return kg
    return w


# Get patient weight in grams
def get_weight_g(ds):
    # Convert kg to grams
    return get_weight_kg(ds) * 1000.0


# Get patient height in meters
def get_height_m(ds):
    # Read patient height from PatientSize
    h = safe_float(getattr(ds, "PatientSize", None))

    # Fail if invalid
    if h is None or h <= 0:
        raise ValueError("Missing or invalid PatientSize")

    return h


# Get patient height in centimeters
def get_height_cm(ds):
    # Convert meters to cm
    return get_height_m(ds) * 100.0


# Normalize patient sex code
def get_patient_sex(ds):
    # Reject absent or invalid values rather than disguising them as "O".
    sex = str(getattr(ds, "PatientSex", "") or "").strip().upper()
    if sex not in ("M", "F", "O"):
        raise ValueError("Missing or invalid PatientSex")
    return sex


# Get injected radionuclide dose in becquerels
def get_total_dose_bq(ds):
    # Get radiopharmaceutical metadata block
    rp = get_radiopharm_item(ds)

    # Read total dose
    dose = safe_float(getattr(rp, "RadionuclideTotalDose", None))

    # Fail if missing or invalid
    if dose is None or dose <= 0:
        raise ValueError("Missing or invalid RadionuclideTotalDose")

    # If it looks too small, assume MBq and convert to Bq
    if dose < 1e4:
        dose *= 1e6

    return dose


# Get radionuclide half-life in seconds
def get_half_life_s(ds):
    # Get radiopharmaceutical metadata block
    rp = get_radiopharm_item(ds)

    # Read half-life
    hl = safe_float(getattr(rp, "RadionuclideHalfLife", None))

    # Fail if missing or invalid
    if hl is None or hl <= 0:
        raise ValueError("Missing or invalid RadionuclideHalfLife")

    return hl


# Determine injection start datetime
def get_radiopharm_start_datetime(ds, reference_dt=None):
    rp = get_radiopharm_item(ds)
    half_life_s = get_half_life_s(ds)
    full_dt = parse_dicom_dt(getattr(rp, "RadiopharmaceuticalStartDateTime", None))
    start_time = getattr(rp, "RadiopharmaceuticalStartTime", None)

    if reference_dt is None:
        reference_dt = parse_dicom_dt(getattr(ds, "AcquisitionDateTime", None))
        if reference_dt is None:
            reference_dt = combine_da_tm(getattr(ds, "AcquisitionDate", None),
                                          getattr(ds, "AcquisitionTime", None))
    if reference_dt is None:
        raise ValueError("Missing decay-correction reference datetime")

    if full_dt is not None:
        delta_s = (reference_dt - full_dt).total_seconds()
        if -3600.0 <= delta_s < 2.0 * half_life_s:
            return full_dt
        if half_life_s >= 41400.0:
            raise ValueError("Unreliable administration date for long-lived radionuclide")
        warnings.warn("Administration date is inconsistent; using reference date and administration time")
        start_time = full_dt.strftime("%H%M%S.%f")
    else:
        if not start_time:
            raise ValueError("Missing radiopharmaceutical start date/time")
        if half_life_s >= 41400.0:
            raise ValueError("Full administration datetime required for long-lived radionuclide")
        warnings.warn("Only administration time available; reference date is assumed")

    t = parse_dicom_tm(start_time)
    if t is None:
        raise ValueError("Invalid radiopharmaceutical administration time")
    start_dt = reference_dt.replace(hour=t[0], minute=t[1], second=t[2], microsecond=t[3])
    if (reference_dt - start_dt).total_seconds() < -3600.0:
        start_dt -= timedelta(days=1)
        warnings.warn("Administration occurred before midnight; using previous day")
    return start_dt


# Compute average count-rate time correction
def average_count_rate_time_s(half_life_s, frame_duration_s):
    # Decay constant
    lam = math.log(2.0) / half_life_s

    # Return average count rate time
    return (1.0 / lam) * math.log((lam * frame_duration_s) / (1.0 - math.exp(-lam * frame_duration_s)))


# Compute reference time used for decay correction START
def get_reference_time_start(ds):
    manufacturer = str(getattr(ds, "Manufacturer", "") or "").upper()
    is_ge = "GE" in manufacturer or "GEMS" in manufacturer
    is_siemens = "SIEMENS" in manufacturer

    # The private scan-start tags are valid for their own manufacturers only.
    if is_ge:
        ge_dt = parse_private_dt(ds, (0x0009, 0x100D))
        if ge_dt is not None:
            return ge_dt
    if is_siemens:
        siemens_dt = parse_private_dt(ds, (0x0071, 0x1022))
        if siemens_dt is not None:
            return siemens_dt

    acq_dt = parse_dicom_dt(getattr(ds, "AcquisitionDateTime", None))
    if acq_dt is None:
        acq_dt = combine_da_tm(getattr(ds, "AcquisitionDate", None),
                                getattr(ds, "AcquisitionTime", None))
    if acq_dt is None:
        raise ValueError("Missing acquisition datetime")

    series_dt = combine_da_tm(getattr(ds, "SeriesDate", None),
                               getattr(ds, "SeriesTime", None))
    if series_dt is not None and series_dt == acq_dt:
        return acq_dt

    frame_ref_ms = safe_float(getattr(ds, "FrameReferenceTime", None))
    if frame_ref_ms is None or frame_ref_ms < 0:
        raise ValueError("Missing or invalid FrameReferenceTime")
    frame_ref_s = frame_ref_ms / 1000.0

    # GE's acquisition timestamp needs no average-count-rate offset.
    if is_ge:
        return acq_dt - timedelta(seconds=frame_ref_s)

    frame_duration_ms = safe_float(getattr(ds, "ActualFrameDuration", None))
    if frame_duration_ms is None or frame_duration_ms <= 0:
        raise ValueError("Missing or invalid ActualFrameDuration")
    half_life_s = get_half_life_s(ds)
    tave_s = average_count_rate_time_s(half_life_s, frame_duration_ms / 1000.0)
    return acq_dt + timedelta(seconds=tave_s - frame_ref_s)


# Lean body mass using James formula variant
def lbm_james128_kg(weight_kg, height_cm, sex):
    male = 1.10 * weight_kg - 128.0 * (weight_kg / height_cm) ** 2
    female = 1.07 * weight_kg - 148.0 * (weight_kg / height_cm) ** 2
    if sex == "M":
        return male
    if sex == "F":
        return female
    return 0.5 * (male + female)


# Lean body mass using Morgan formula variant
def lbm_morgan_kg(weight_kg, height_cm, sex):
    male = 1.10 * weight_kg - 120.0 * (weight_kg / height_cm) ** 2
    female = 1.07 * weight_kg - 148.0 * (weight_kg / height_cm) ** 2
    if sex == "M":
        return male
    if sex == "F":
        return female
    return 0.5 * (male + female)


# Lean body mass using Janmahasatian formula
def lbm_janma_kg(weight_kg, height_cm, sex):
    bmi = weight_kg / ((height_cm / 100.0) ** 2)
    male = 9270.0 * weight_kg / (6680.0 + 216.0 * bmi)
    female = 9270.0 * weight_kg / (8780.0 + 244.0 * bmi)
    if sex == "M":
        return male
    if sex == "F":
        return female
    return 0.5 * (male + female)


# Ideal body weight formula
def ibw_kg(height_cm, sex):
    male = 48.0 + 1.06 * (height_cm - 152.0)
    female = 45.5 + 0.91 * (height_cm - 152.0)
    if sex == "M":
        return male
    if sex == "F":
        return female
    return 0.5 * (male + female)


# Body surface area, returned in cm^2
def bsa_cm2(weight_kg, height_cm):
    bsa_m2 = 0.007184 * (height_cm ** 0.725) * (weight_kg ** 0.425)
    return bsa_m2 * 10000.0


# Convert BQML PET values to SUVbw
def convert_bqml_to_suvbw(datasets, values):
    out = np.empty_like(values, dtype=np.float64)
    # Use each frame's dose, half-life, and reference/administration times.
    for z, ds in enumerate(datasets):
        weight_g = get_weight_g(ds)
        dose_bq_admin = get_total_dose_bq(ds)
        half_life_s = get_half_life_s(ds)
        lam = math.log(2.0) / half_life_s
        decay_correction = str(getattr(ds, "DecayCorrection", "") or "").upper().strip()

        if decay_correction == "ADMIN":
            dose_at_ref = dose_bq_admin
        elif decay_correction == "START":
            tref = get_reference_time_start(ds)
            tadm = get_radiopharm_start_datetime(ds, reference_dt=tref)
            dose_at_ref = dose_bq_admin * math.exp(-lam * (tref - tadm).total_seconds())
        elif decay_correction == "NONE":
            acq_dt = parse_dicom_dt(getattr(ds, "AcquisitionDateTime", None))
            if acq_dt is None:
                acq_dt = combine_da_tm(getattr(ds, "AcquisitionDate", None),
                                       getattr(ds, "AcquisitionTime", None))
            if acq_dt is None:
                raise ValueError("Missing acquisition time for DecayCorrection=NONE")
            frame_duration_ms = safe_float(getattr(ds, "ActualFrameDuration", None))
            if frame_duration_ms is None or frame_duration_ms <= 0:
                raise ValueError("Missing or invalid ActualFrameDuration for DecayCorrection=NONE")
            tave_s = average_count_rate_time_s(half_life_s, frame_duration_ms / 1000.0)
            tref = acq_dt + timedelta(seconds=tave_s)
            tadm = get_radiopharm_start_datetime(ds, reference_dt=tref)
            dose_at_ref = dose_bq_admin * math.exp(-lam * (tref - tadm).total_seconds())
        else:
            raise NotImplementedError(f"Unsupported DecayCorrection: {decay_correction}")

        if dose_at_ref <= 0 or not math.isfinite(dose_at_ref):
            raise ValueError("Invalid decay-corrected radionuclide dose")
        out[z] = values[z] * (weight_g / dose_at_ref)

    return out


# Convert GML values to SUVbw
def convert_gml_to_suvbw(ds0, values):
    suv_type = str(getattr(ds0, "SUVType", "") or "").upper().strip()
    if suv_type == "":
        suv_type = "BW"

    if suv_type == "BW":
        return values

    weight_kg = get_weight_kg(ds0)
    height_cm = get_height_cm(ds0)
    sex = get_patient_sex(ds0)

    if suv_type == "LBM":
        factor = lbm_morgan_kg(weight_kg, height_cm, sex)
    elif suv_type == "LBMJAMES128":
        factor = lbm_james128_kg(weight_kg, height_cm, sex)
    elif suv_type == "LBMJANMA":
        factor = lbm_janma_kg(weight_kg, height_cm, sex)
    elif suv_type == "IBW":
        factor = ibw_kg(height_cm, sex)
    else:
        raise NotImplementedError(f"Unsupported SUVType in GML: {suv_type}")

    if factor <= 0:
        raise ValueError(f"Computed non-positive normalization factor for SUVType={suv_type}")

    return values * (weight_kg / factor)


# Convert CM2ML values to SUVbw
def convert_cm2ml_to_suvbw(ds0, values):
    suv_type = str(getattr(ds0, "SUVType", "") or "").upper().strip()

    # Allow missing SUVType, but if present it should be BSA
    if suv_type not in ["", "BSA"]:
        raise ValueError(f"Unexpected SUVType for CM2ML: {suv_type}")

    weight_kg = get_weight_kg(ds0)
    weight_g = weight_kg * 1000.0
    height_cm = get_height_cm(ds0)
    bsa = bsa_cm2(weight_kg, height_cm)

    if bsa <= 0:
        raise ValueError("Computed non-positive BSA")

    return values * (weight_g / bsa)


# Convert Philips CNTS values to SUVbw
def convert_cnts_to_suvbw(datasets, values):
    out = np.empty_like(values, dtype=np.float64)
    for z, ds in enumerate(datasets):
        if "PHILIPS" not in str(getattr(ds, "Manufacturer", "") or "").upper():
            raise ValueError("CNTS scaling requires Philips manufacturer")
        suv_sf_tag = ds.get((0x7053, 0x1000))
        ac_sf_tag = ds.get((0x7053, 0x1009))
        suv_sf = safe_float(suv_sf_tag.value if suv_sf_tag is not None else None)
        ac_sf = safe_float(ac_sf_tag.value if ac_sf_tag is not None else None)

        # Prefer activity concentration when available (IBSI v3).
        if ac_sf is not None and ac_sf != 0:
            out[z] = convert_bqml_to_suvbw([ds], values[z:z+1] * ac_sf)[0]
        elif suv_sf is not None and suv_sf != 0:
            suv_type = str(getattr(ds, "SUVType", "") or "").upper().strip()
            if suv_type not in ("", "BW"):
                raise ValueError("Philips direct SUV factor does not specify SUVbw")
            out[z] = values[z] * suv_sf
        else:
            raise ValueError("CNTS without valid Philips scale factors")
    return out


# Main converter that sends PET values to SUVbw depending on DICOM Units
def convert_to_suvbw(datasets, volume, slopes, intercepts):
    ds0 = datasets[0]

    if np.any(np.abs(intercepts) > 1e-12):
        raise ValueError("RescaleIntercept is not zero. Excluding dataset.")

    values = apply_rescale(volume, slopes, intercepts)
    units = str(getattr(ds0, "Units", "") or "").upper().strip()

    if units == "BQML":
        return convert_bqml_to_suvbw(datasets, values)
    if units == "GML":
        return convert_gml_to_suvbw(ds0, values)
    if units == "CM2ML":
        return convert_cm2ml_to_suvbw(ds0, values)
    if units == "CNTS":
        return convert_cnts_to_suvbw(datasets, values)

    raise NotImplementedError(f"Unsupported Units: {units}")


# Enhanced PET uses per-frame functional groups and real-world mappings.
# The conventional PET calculation functions above are deliberately unchanged.
ENHANCED_PET_SOP_UID = "1.2.840.10008.5.1.4.1.1.130"


class SUVValidationError(ValueError):
    """Invalid PET metadata prevents a valid SUVbw result."""


def is_enhanced_pet_series(pt_dir):
    """Use the SOP class rather than frame count or a DRO directory name."""
    paths = find_dicom_files(pt_dir)
    sop_classes = {
        str(getattr(pydicom.dcmread(path, stop_before_pixels=True, force=True),
                    "SOPClassUID", ""))
        for path in paths
    }
    if ENHANCED_PET_SOP_UID in sop_classes and len(sop_classes) != 1:
        raise ValueError("Mixed Enhanced and conventional PET objects in one PT directory")
    return sop_classes == {ENHANCED_PET_SOP_UID}


def enhanced_group_attribute(dataset, per_frame, shared, attribute):
    """Find frame-specific information first, then shared, then root metadata.

    Some functional-group values are stored in a one-item nested sequence.
    Avoid searching inside image mappings when looking for unrelated metadata.
    """
    for group in (per_frame, shared):
        if group is None:
            continue
        direct = getattr(group, attribute, None)
        if direct is not None and direct != "":
            return direct
        for elem in group:
            if getattr(elem, "VR", None) != "SQ":
                continue
            for item in elem.value:
                found = getattr(item, attribute, None)
                if found is not None and found != "":
                    return found
    return getattr(dataset, attribute, None)


def enhanced_world_mappings(dataset, per_frame, shared):
    """Per-frame RWVM takes precedence over shared and root RWVM."""
    for group in (per_frame, shared, dataset):
        if group is None:
            continue
        mapping = getattr(group, "RealWorldValueMappingSequence", None)
        if mapping:
            return mapping
    return None


def enhanced_unit(mapping):
    units = getattr(mapping, "MeasurementUnitsCodeSequence", None)
    if not units:
        return None
    code_item = units[0]
    value = (getattr(code_item, "CodeValue", None)
             or getattr(code_item, "LongCodeValue", None)
             or getattr(code_item, "URNCodeValue", None))
    if not value:
        return None
    code = str(value).replace(" ", "").lower()
    if code in ("bq/ml", "bqml"):
        return "BQML", ""
    if code in ("g/ml{suvbw}", "g/ml{suvbodyweight}"):
        return "GML", "BW"
    if code in ("g/ml{suvlbm}", "g/ml{suvlbmmorgan}"):
        return "GML", "LBM"
    if code == "g/ml{suvlbmjames128}":
        return "GML", "LBMJAMES128"
    if code in ("g/ml{suvlbmjanma}", "g/ml{suvlbmjanmahasatian}"):
        return "GML", "LBMJANMA"
    if code == "g/ml{suvibw}":
        return "GML", "IBW"
    if code in ("cm2/ml{suvbsa}", "cm^2/ml{suvbsa}"):
        return "CM2ML", "BSA"
    return None


def enhanced_apply_world_mapping(pixel, mapping):
    """Apply the RWVM slope/intercept or a complete real-world-value LUT."""
    slope = safe_float(getattr(mapping, "RealWorldValueSlope", None))
    intercept = safe_float(getattr(mapping, "RealWorldValueIntercept", None))
    if slope is not None and intercept is not None:
        if not math.isfinite(slope) or not math.isfinite(intercept):
            raise ValueError("Invalid Enhanced PET real-world mapping parameters")
        return pixel * slope + intercept
    lut = getattr(mapping, "RealWorldValueLUTData", None)
    if lut is None:
        raise ValueError("Enhanced PET mapping lacks slope/intercept or LUT")
    first = getattr(mapping, "RealWorldValueFirstValueMapped", None)
    last = getattr(mapping, "RealWorldValueLastValueMapped", None)
    if first is None or last is None:
        first = getattr(mapping, "DoubleFloatRealWorldValueFirstValueMapped", None)
        last = getattr(mapping, "DoubleFloatRealWorldValueLastValueMapped", None)
    if first is None or last is None:
        raise ValueError("Enhanced PET LUT missing mapped-value bounds")
    first = float(first)
    last = float(last)
    lut_values = np.asarray(lut, dtype=np.float64).ravel()
    if (not np.isfinite(first) or not np.isfinite(last)
            or last < first or not float(first).is_integer()
            or lut_values.size != int(last - first + 1)
            or not np.all(np.isfinite(lut_values))):
        raise ValueError("Invalid Enhanced PET real-world-value LUT")
    indices = pixel - first
    if (np.any(indices < 0) or np.any(indices >= lut_values.size)
            or not np.all(indices == np.rint(indices))):
        raise ValueError("Enhanced PET voxel outside real-world-value LUT range")
    return lut_values[indices.astype(np.int64)]


def enhanced_mapping_for_frame(ds, per_frame, shared):
    mapping_seq = enhanced_world_mappings(ds, per_frame, shared)
    if mapping_seq:
        # A higher-priority mapping must have sufficient scaling information.
        valid = []
        for m in mapping_seq:
            unit = enhanced_unit(m)
            slope = safe_float(getattr(m, "RealWorldValueSlope", None))
            intercept = safe_float(getattr(m, "RealWorldValueIntercept", None))
            has_linear = (slope is not None and intercept is not None
                          and math.isfinite(slope) and math.isfinite(intercept))
            has_lut = (getattr(m, "RealWorldValueLUTData", None) is not None
                       and (getattr(m, "RealWorldValueFirstValueMapped", None) is not None
                            or getattr(m, "DoubleFloatRealWorldValueFirstValueMapped", None) is not None))
            if unit is not None and (has_linear or has_lut):
                valid.append((unit, m))
        # Prefer direct SUVbw, then another SUV type, then activity concentration.
        for wanted in ("GML_BW", "GML", "CM2ML", "BQML"):
            for unit, m in valid:
                if unit is None:
                    continue
                kind, suv_type = unit
                if (wanted == "GML_BW" and kind == "GML" and suv_type == "BW"
                        or wanted == "GML" and kind == "GML" and suv_type != "BW"
                        or wanted == kind and wanted in ("CM2ML", "BQML")):
                    return unit, m

    # IBSI fallback: Enhanced PET Pixel Value Transformation Sequence.
    for group in (per_frame, shared):
        seq = getattr(group, "PixelValueTransformationSequence", None) if group is not None else None
        if not seq:
            continue
        item = seq[0]
        kind = str(getattr(item, "RescaleType", "") or "").upper().strip()
        if kind not in ("BQML", "GML", "CM2ML"):
            continue
        slope = safe_float(getattr(item, "RescaleSlope", None))
        intercept = safe_float(getattr(item, "RescaleIntercept", None))
        if slope is None or intercept is None:
            continue
        suv_type = (str(getattr(ds, "SUVType", "") or "").upper().strip()
                    if kind == "GML" else "BSA" if kind == "CM2ML" else "")
        if kind == "GML" and suv_type == "":
            suv_type = "BW"
        return (kind, suv_type), SimpleNamespace(RealWorldValueSlope=slope,
                                                RealWorldValueIntercept=intercept)
    raise ValueError("Enhanced PET lacks a usable real-world mapping or pixel transformation")


def load_enhanced_pet_suvbw(pt_dir):
    """Return spatially sorted frame metadata, SUVbw volume, and original object.

    DICOM Per-frame Functional Groups Sequence items index the raw frames.
    The original object is retained for writing an Enhanced PET DICOM output.
    """
    paths = find_dicom_files(pt_dir)
    if len(paths) != 1:
        raise ValueError("Enhanced PET expects one multiframe object per case")
    source = pydicom.dcmread(paths[0], force=True)
    if str(getattr(source, "SOPClassUID", "")) != ENHANCED_PET_SOP_UID:
        raise ValueError("Expected Enhanced PET SOP Class")
    pixels = source.pixel_array.astype(np.float64)
    if pixels.ndim == 2:
        pixels = pixels[np.newaxis, :, :]
    if pixels.ndim != 3:
        raise ValueError("Enhanced PET pixel array must be [frame, row, column]")
    nframes = int(getattr(source, "NumberOfFrames", pixels.shape[0]))
    if nframes != pixels.shape[0]:
        raise ValueError("Enhanced PET NumberOfFrames does not match PixelData")
    per_frames = getattr(source, "PerFrameFunctionalGroupsSequence", None)
    if not per_frames or len(per_frames) != nframes:
        raise ValueError("Missing or inconsistent Per-Frame Functional Groups Sequence")
    shared_frames = getattr(source, "SharedFunctionalGroupsSequence", None)
    shared = shared_frames[0] if shared_frames else None
    metadata = []
    suv_frames = []
    for index in range(nframes):
        pf = per_frames[index]
        orient = enhanced_group_attribute(source, pf, shared, "ImageOrientationPatient")
        position = enhanced_group_attribute(source, pf, shared, "ImagePositionPatient")
        spacing = enhanced_group_attribute(source, pf, shared, "PixelSpacing")
        if orient is None or position is None or spacing is None:
            raise ValueError(f"Enhanced frame {index}: missing image geometry")
        unit_and_type, mapping = enhanced_mapping_for_frame(source, pf, shared)
        kind, suv_type = unit_and_type
        real_values = enhanced_apply_world_mapping(pixels[index], mapping)
        frame = SimpleNamespace(
            _frame_index=index,
            ImageOrientationPatient=orient,
            ImagePositionPatient=position,
            PixelSpacing=spacing,
            InstanceNumber=index + 1,
            PatientWeight=getattr(source, "PatientWeight", None),
            PatientSize=getattr(source, "PatientSize", None),
            PatientSex=getattr(source, "PatientSex", None),
            Units=kind, SUVType=suv_type,
            RadiopharmaceuticalInformationSequence=enhanced_group_attribute(
                source, pf, shared, "RadiopharmaceuticalInformationSequence"),
        )
        if kind == "BQML":
            # Enhanced PET uses full administration datetime, never time-only fallback.
            rp = get_radiopharm_item(frame)
            if parse_dicom_dt(getattr(rp, "RadiopharmaceuticalStartDateTime", None)) is None:
                raise ValueError("Enhanced PET missing full RadiopharmaceuticalStartDateTime")
            corrected = str(enhanced_group_attribute(source, pf, shared,
                                                    "DecayCorrected") or "").upper().strip()
            if corrected not in ("YES", "NO"):
                raise ValueError("Enhanced PET missing or invalid DecayCorrected")
            if corrected == "YES":
                reference = parse_dicom_dt(enhanced_group_attribute(
                    source, pf, shared, "DecayCorrectionDateTime"))
                if reference is None:
                    raise ValueError("Enhanced PET missing DecayCorrectionDateTime")
            else:
                reference = parse_dicom_dt(enhanced_group_attribute(
                    source, pf, shared, "FrameReferenceDateTime"))
                if reference is None:
                    start = parse_dicom_dt(enhanced_group_attribute(
                        source, pf, shared, "FrameAcquisitionDateTime"))
                    duration = safe_float(enhanced_group_attribute(
                        source, pf, shared, "FrameAcquisitionDuration"))
                    if start is None or duration is None or duration <= 0:
                        raise ValueError("Enhanced PET lacks measurement datetime and frame duration")
                    reference = start + timedelta(seconds=average_count_rate_time_s(
                        get_half_life_s(frame), duration / 1000.0))
            frame.DecayCorrection = "START" if corrected == "YES" else "NONE"
            admin = get_radiopharm_start_datetime(frame, reference_dt=reference)
            dose = get_total_dose_bq(frame)
            half_life = get_half_life_s(frame)
            dose_ref = dose * math.exp(-math.log(2.0) / half_life *
                                      (reference - admin).total_seconds())
            if not math.isfinite(dose_ref) or dose_ref <= 0:
                raise ValueError("Enhanced PET invalid decay-corrected dose")
            result = real_values * (get_weight_g(frame) / dose_ref)
        elif kind == "GML":
            frame.DecayCorrection = str(getattr(source, "DecayCorrection", "") or "")
            result = convert_gml_to_suvbw(frame, real_values)
        elif kind == "CM2ML":
            frame.DecayCorrection = str(getattr(source, "DecayCorrection", "") or "")
            result = convert_cm2ml_to_suvbw(frame, real_values)
        else:
            raise ValueError(f"Unsupported Enhanced PET units: {kind}")
        if not np.all(np.isfinite(result)):
            raise ValueError("Non-finite SUVbw in Enhanced PET frame")
        metadata.append(frame)
        suv_frames.append(result)
    # Geometry sorting must reorder pixel frames and metadata together.
    sorted_metadata = sort_pet_slices(metadata)
    volume = np.stack([suv_frames[frame._frame_index] for frame in sorted_metadata], axis=0)
    return sorted_metadata, volume, source


def save_enhanced_suv_dicom_series(case_dir, output_root):
    """Write the converted frames as one Enhanced PET object in original frame order."""
    case_name = os.path.basename(case_dir.rstrip(os.sep))
    metadata, suv_volume, source = load_enhanced_pet_suvbw(os.path.join(case_dir, "PT"))
    case_out_dir = os.path.join(output_root, case_name)
    os.makedirs(case_out_dir, exist_ok=True)
    original_order = np.empty_like(suv_volume)
    for sorted_index, frame in enumerate(metadata):
        original_order[frame._frame_index] = suv_volume[sorted_index]
    max_suv = float(np.max(original_order))
    common_slope = max(max_suv / 65535.0, 1e-6) if max_suv > 0 else 1.0
    stored = np.clip(np.rint(original_order / common_slope), 0, 65535).astype(np.uint16)
    new_ds = copy.deepcopy(source)
    new_ds.PixelData = stored.tobytes()
    new_ds.BitsAllocated = 16
    new_ds.BitsStored = 16
    new_ds.HighBit = 15
    new_ds.PixelRepresentation = 0
    new_ds.SamplesPerPixel = 1
    new_ds.PhotometricInterpretation = "MONOCHROME2"
    new_ds.SeriesInstanceUID = generate_uid()
    new_ds.SOPInstanceUID = generate_uid()
    if hasattr(new_ds, "file_meta") and new_ds.file_meta is not None:
        new_ds.file_meta.MediaStorageSOPInstanceUID = new_ds.SOPInstanceUID
    new_ds.SeriesDescription = "SUVbw Corrected PET"
    new_ds.ImageType = ["DERIVED", "SECONDARY"]
    new_ds.Units = "GML"
    new_ds.SUVType = "BW"
    new_ds.RescaleSlope = float(common_slope)
    new_ds.RescaleIntercept = 0.0
    new_ds.SeriesNumber = int(getattr(source, "SeriesNumber", 0) or 0) + 1000
    # Replace the source units/mappings. Otherwise a reader would apply the
    # source Bq/ml mapping to the newly stored SUVbw pixel values.
    for attr in ("RealWorldValueMappingSequence", "PixelValueTransformationSequence"):
        if hasattr(new_ds, attr):
            delattr(new_ds, attr)
    from pydicom.dataset import Dataset
    from pydicom.sequence import Sequence
    shared = getattr(new_ds, "SharedFunctionalGroupsSequence", None)
    if not shared:
        shared = Sequence([Dataset()])
        new_ds.SharedFunctionalGroupsSequence = shared
    shared_item = shared[0]
    if hasattr(shared_item, "PixelValueTransformationSequence"):
        del shared_item.PixelValueTransformationSequence
    for group in new_ds.PerFrameFunctionalGroupsSequence:
        for attr in ("RealWorldValueMappingSequence", "PixelValueTransformationSequence"):
            if hasattr(group, attr):
                delattr(group, attr)
    code = Dataset()
    code.CodeValue = "g/ml{SUVbw}"
    code.CodingSchemeDesignator = "UCUM"
    code.CodeMeaning = "Standardized uptake value, body weight"
    mapping = Dataset()
    mapping.RealWorldValueSlope = float(common_slope)
    mapping.RealWorldValueIntercept = 0.0
    mapping.MeasurementUnitsCodeSequence = Sequence([code])
    mapping.RealWorldValueFirstValueMapped = 0
    mapping.RealWorldValueLastValueMapped = 65535
    shared_item.RealWorldValueMappingSequence = Sequence([mapping])
    path = os.path.join(case_out_dir, "SUV_0001.dcm")
    new_ds.save_as(path, write_like_original=False)
    return case_name, case_out_dir, suv_volume.shape, common_slope


# Load a NIfTI mask and align it to PET shape
def load_nifti_mask(mask_path, pet_shape=None, erode_voxels=0, datasets=None):
    nii = nib.load(mask_path)
    mask = nii.get_fdata() > 0

    if pet_shape is not None and datasets is not None:
        ds0 = datasets[0]
        orientation = np.asarray(ds0.ImageOrientationPatient, dtype=float)
        spacing = np.asarray(ds0.PixelSpacing, dtype=float)
        positions = np.asarray([ds.ImagePositionPatient for ds in datasets], dtype=float)
        if orientation.shape != (6,) or spacing.shape != (2,) or positions.shape != (len(datasets), 3):
            raise ValueError("Missing PET geometry needed for NIfTI alignment")
        if np.any(spacing <= 0) or not np.all(np.isfinite(positions)):
            raise ValueError("Invalid PET geometry")

        col_step = orientation[:3] * spacing[1]
        row_step = orientation[3:] * spacing[0]
        if len(datasets) > 1:
            slice_step = positions[1] - positions[0]
            expected = positions[0] + np.arange(len(datasets))[:, None] * slice_step
            if not np.allclose(positions, expected, atol=0.05, rtol=0):
                raise ValueError("PET slices are not on a regular spatial grid")
        else:
            depth_mm = safe_float(getattr(ds0, "SpacingBetweenSlices", None))
            if depth_mm is None or depth_mm <= 0:
                depth_mm = safe_float(getattr(ds0, "SliceThickness", None))
            if depth_mm is None or depth_mm <= 0:
                raise ValueError("Missing slice spacing for PET mask alignment")
            slice_step = np.cross(orientation[:3], orientation[3:]) * depth_mm

        # PET numpy order is [slice, row, column]; DICOM positions are LPS.
        pet_affine_lps = np.eye(4)
        pet_affine_lps[:3, :3] = np.column_stack((slice_step, row_step, col_step))
        pet_affine_lps[:3, 3] = positions[0]
        lps_to_ras = np.diag([-1.0, -1.0, 1.0, 1.0])
        try:
            mapping = np.linalg.inv(nii.affine) @ lps_to_ras @ pet_affine_lps
        except np.linalg.LinAlgError as exc:
            raise ValueError("Non-invertible NIfTI affine") from exc

        # Alignment must be an exact index permutation/flip/shift, no interpolation.
        matrix = mapping[:3, :3]
        origin = mapping[:3, 3]
        rounded = np.rint(matrix).astype(int)
        offset = np.rint(origin).astype(int)
        if (not np.allclose(matrix, rounded, atol=1e-3, rtol=0)
                or not np.allclose(origin, offset, atol=1e-3, rtol=0)
                or not np.all(np.sum(np.abs(rounded), axis=0) == 1)
                or not np.all(np.sum(np.abs(rounded), axis=1) == 1)):
            raise ValueError("PET/NIfTI geometry cannot be aligned without interpolation")

        idx = []
        for axis in range(3):
            source_axis = int(np.flatnonzero(rounded[axis])[0])
            vector = offset[axis] + rounded[axis, source_axis] * np.arange(pet_shape[source_axis])
            if vector.min() < 0 or vector.max() >= mask.shape[axis]:
                raise ValueError("PET voxels extend outside the NIfTI mask grid")
            idx.append(vector)
        # Select only corresponding voxels and permute axes to PET [z, y, x].
        nifti_source_axes = [int(np.flatnonzero(rounded[a])[0]) for a in range(3)]
        aligned = np.transpose(mask[np.ix_(*idx)], axes=np.argsort(nifti_source_axes))
    elif pet_shape is not None:
        raise ValueError("PET DICOM metadata required for verified NIfTI alignment")
    else:
        aligned = mask

    if erode_voxels > 0:
        aligned = binary_erosion(aligned, iterations=erode_voxels)
    return aligned


# Compute SUV summary metrics inside the mask
def calculate_suv_values(suv_volume, mask):
    if suv_volume.shape != mask.shape:
        raise ValueError(f"Shape mismatch: PET {suv_volume.shape} vs mask {mask.shape}")

    vals = suv_volume[mask]

    if vals.size == 0:
        raise ValueError("Mask is empty")

    return {
        "Nvox": int(vals.size),
        "SUVmin": round(float(np.min(vals)), 2),
        "SUVmedian": round(float(np.median(vals)), 2),
        "SUVmax": round(float(np.max(vals)), 2),
        "SUVmax_validation": float(np.max(vals)),
        "SUVmean": float(np.mean(vals)),
    }


# Extract a small metadata summary for reporting
def summarize_case_metadata(ds):
    rp = get_radiopharm_item(ds)
    return {
        "PatientName": str(getattr(ds, "PatientName", "") or ""),
        "PatientID": str(getattr(ds, "PatientID", "") or ""),
        "Units": str(getattr(ds, "Units", "") or ""),
        "SUVType": str(getattr(ds, "SUVType", "") or ""),
        "DecayCorrection": str(getattr(ds, "DecayCorrection", "") or ""),
        "PatientWeight": safe_float(getattr(ds, "PatientWeight", None)),
        "PatientSize": safe_float(getattr(ds, "PatientSize", None)),
        "RadionuclideTotalDose": safe_float(getattr(rp, "RadionuclideTotalDose", None)),
        "RadionuclideHalfLife": safe_float(getattr(rp, "RadionuclideHalfLife", None)),
    }


# Process one case folder from start to finish
def process_case(case_dir, erode_voxels=0):
    # Extract case folder name
    case_name = os.path.basename(case_dir.rstrip(os.sep))

    # Build PET folder path
    pt_dir = os.path.join(case_dir, "PT")

    # Build mask folder path
    mask_dir = os.path.join(case_dir, "mask")

    # Find mask file
    mask_path = find_mask_file(mask_dir)

    # Convert using the SOP class; preserve the existing classic-PET pathway.
    if is_enhanced_pet_series(pt_dir):
        datasets, suv, _ = load_enhanced_pet_suvbw(pt_dir)
    else:
        datasets, volume, slopes, intercepts = load_pet_series(pt_dir)
        try:
            suv = convert_to_suvbw(datasets, volume, slopes, intercepts)
        except (ValueError, NotImplementedError) as exc:
            raise SUVValidationError(str(exc)) from exc

    # Load and align NIfTI mask
    mask = load_nifti_mask(
        mask_path,
        pet_shape=suv.shape,
        erode_voxels=erode_voxels,
        datasets=datasets
    )

    # Compute SUV metrics inside the mask
    metrics = calculate_suv_values(suv, mask)

    # Collect metadata summary
    meta = summarize_case_metadata(datasets[0])

    # Return one output row
    return {
        "case": case_name,
        "mask_file": os.path.basename(mask_path),
        "erode_voxels": erode_voxels,
        "Nvox": metrics["Nvox"],
        "Units": meta["Units"],
        "SUVType": meta["SUVType"],
        "DecayCorrection": meta["DecayCorrection"],
        "SUVmin": metrics["SUVmin"],
        "SUVmedian": metrics["SUVmedian"],
        "SUVmax": metrics["SUVmax"],
        "PatientDir": case_name,
        "PatientName": meta["PatientName"],
        "PatientID": meta["PatientID"],
        "SUVmax_validation": metrics["SUVmax_validation"],
        "SUVmean": metrics["SUVmean"],
    }


# Process every case folder and save summary CSV
def run_all_cases(root_dir, output_csv, erode_voxels=0):
    results = []

    # Make sure output folder exists
    out_parent = os.path.dirname(output_csv)
    if out_parent:
        os.makedirs(out_parent, exist_ok=True)

    case_dirs = find_case_dirs(root_dir)

    for case_dir in case_dirs:
        case_name = os.path.basename(case_dir)

        try:
            row = process_case(case_dir, erode_voxels=erode_voxels)
            results.append(row)
            print(
                f"{case_name}: "
                f"Units={row['Units']}, "
                f"DecayCorrection={row['DecayCorrection']}, "
                f"Nvox={row['Nvox']}, "
                f"SUVmin={row['SUVmin']:.2f}, "
                f"SUVmedian={row['SUVmedian']:.2f}, "
                f"SUVmax={row['SUVmax']:.2f}"
            )
        except Exception as e:
            # ERROR is reserved for rejected SUV metadata, not unreadable images,
            # mask-alignment failures, or missing dependencies.
            status = "ERROR" if isinstance(e, SUVValidationError) else "FAILED"
            patient_name = ""
            patient_id = ""
            try:
                failed_pt_dir = os.path.join(case_dir, "PT")
                failed_dicom_files = find_dicom_files(failed_pt_dir)
                failed_ds = pydicom.dcmread(
                    failed_dicom_files[0], stop_before_pixels=True, force=True
                )
                patient_name = str(getattr(failed_ds, "PatientName", "") or "")
                patient_id = str(getattr(failed_ds, "PatientID", "") or "")
            except Exception:
                pass
            results.append({"case": case_name, "mask_file": "",
                            "erode_voxels": erode_voxels, "Nvox": "",
                            "Units": "", "SUVType": "", "DecayCorrection": "",
                            "SUVmin": status, "SUVmedian": status, "SUVmax": status,
                            "PatientDir": case_name, "PatientName": patient_name,
                            "PatientID": patient_id, "SUVmax_validation": status,
                            "SUVmean": status})
            print(f"{case_name}: {status} -> {e}")

    # Save this run's results under a unique filename. An older
    # suvbw_results_standardized.csv may be open in Excel or belong to an
    # earlier script (including one that used erode_voxels=1). Never treat it
    # as this run's calculated results, and never overwrite it silently.
    output_stem, output_ext = os.path.splitext(output_csv)
    run_id = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    actual_output_csv = f"{output_stem}_run_{run_id}{output_ext or '.csv'}"
    # Write the calculated results without overwriting any previous run.
    results_by_case = {row["PatientDir"]: row for row in results}
    output_rows = []

    for case_dir in case_dirs:
        case_name = os.path.basename(case_dir)
        row = results_by_case.get(case_name)

        if row is None:
            row = {
                "PatientDir": case_name,
                "PatientName": "",
                "PatientID": "",
                "SUVmin": "FAILED",
                "SUVmedian": "FAILED",
                "SUVmax_validation": "FAILED",
                "SUVmean": "FAILED",
            }

        if not row["PatientName"] or not row["PatientID"]:
            try:
                pt_dir = os.path.join(case_dir, "PT")
                dicom_files = find_dicom_files(pt_dir)
                ds_meta = pydicom.dcmread(
                    dicom_files[0], stop_before_pixels=True, force=True
                )
                if not row["PatientName"]:
                    row["PatientName"] = str(
                        getattr(ds_meta, "PatientName", "") or ""
                    )
                if not row["PatientID"]:
                    row["PatientID"] = str(
                        getattr(ds_meta, "PatientID", "") or ""
                    )
            except Exception:
                pass

        output_rows.append({
            "PatientDir": row["PatientDir"],
            "PatientName": row["PatientName"],
            "PatientID": row["PatientID"],
            "SUVmin": (f"{row['SUVmin']:.3f}"
                       if isinstance(row["SUVmin"], (int, float, np.number))
                       else row["SUVmin"]),
            "SUVmedian": (f"{row['SUVmedian']:.3f}"
                          if isinstance(row["SUVmedian"], (int, float, np.number))
                          else row["SUVmedian"]),
            "SUVmax": (f"{row['SUVmax_validation']:.3f}"
                       if isinstance(row["SUVmax_validation"], (int, float, np.number))
                       else row["SUVmax_validation"]),
            "SUVmean": (f"{row['SUVmean']:.3f}"
                        if isinstance(row["SUVmean"], (int, float, np.number))
                        else row["SUVmean"]),
        })

    with open(actual_output_csv, "x", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["PatientDir", "PatientName", "PatientID", "SUVmin", "SUVmedian", "SUVmax", "SUVmean"],
            quoting=csv.QUOTE_ALL
        )
        writer.writeheader()
        writer.writerows(output_rows)

    # Verify the newly written results describe this run.
    with open(actual_output_csv, "r", newline="", encoding="utf-8") as f:
        written_results = list(csv.DictReader(f))
    if (
        len(written_results) != len(output_rows)
        or [r["PatientDir"] for r in written_results]
        != [r["PatientDir"] for r in output_rows]
    ):
        raise IOError("New results CSV does not match this run")

    print(f"\nFRESH RESULTS CSV: {os.path.abspath(actual_output_csv)}")
    print(f"Verified {len(written_results)} result rows; "
          f"mask erosion = {erode_voxels}.")
    print(f"Previous fixed-name CSV remains unchanged: {os.path.abspath(output_csv)}")
    return pd.DataFrame(output_rows)


# Save a new DICOM PET series whose values represent SUVbw
def save_suv_dicom_series(case_dir, output_root):
    case_name = os.path.basename(case_dir.rstrip(os.sep))
    pt_dir = os.path.join(case_dir, "PT")

    if is_enhanced_pet_series(pt_dir):
        return save_enhanced_suv_dicom_series(case_dir, output_root)

    datasets, volume, slopes, intercepts = load_pet_series(pt_dir)
    suv_volume = convert_to_suvbw(datasets, volume, slopes, intercepts)

    case_out_dir = os.path.join(output_root, case_name)
    os.makedirs(case_out_dir, exist_ok=True)

    suv_max = float(np.max(suv_volume))
    if suv_max <= 0:
        common_slope = 1.0
    else:
        common_slope = max(suv_max / 65535.0, 1e-6)

    new_series_uid = generate_uid()

    for z, ds in enumerate(datasets):
        new_ds = copy.deepcopy(ds)

        suv_slice = suv_volume[z].astype(np.float64)
        stored = np.round(suv_slice / common_slope)
        stored = np.clip(stored, 0, 65535).astype(np.uint16)

        new_ds.PixelData = stored.tobytes()
        new_ds.Rows, new_ds.Columns = stored.shape
        new_ds.BitsAllocated = 16
        new_ds.BitsStored = 16
        new_ds.HighBit = 15
        new_ds.PixelRepresentation = 0
        new_ds.SamplesPerPixel = 1
        new_ds.PhotometricInterpretation = "MONOCHROME2"

        new_ds.SeriesInstanceUID = new_series_uid
        new_ds.SOPInstanceUID = generate_uid()
        if hasattr(new_ds, "file_meta") and new_ds.file_meta is not None:
            new_ds.file_meta.MediaStorageSOPInstanceUID = new_ds.SOPInstanceUID

        new_ds.SeriesDescription = "SUVbw Corrected PET"
        new_ds.ImageType = ["DERIVED", "SECONDARY"]
        new_ds.Units = "GML"
        new_ds.SUVType = "BW"
        new_ds.RescaleSlope = float(common_slope)
        new_ds.RescaleIntercept = 0.0
        new_ds.SeriesNumber = int(getattr(ds, "SeriesNumber", 0) or 0) + 1000
        new_ds.InstanceNumber = z + 1

        out_path = os.path.join(case_out_dir, f"SUV_{z+1:04d}.dcm")
        new_ds.save_as(out_path, write_like_original=False)

    return case_name, case_out_dir, suv_volume.shape, common_slope



In [15]:
# Loop through everything inside the root directory in sorted order
for name in sorted(os.listdir(ROOT_DIR)):
    # Build the full path for this item
    case_path = os.path.join(ROOT_DIR, name)

    # Only continue if the item is a folder
    if os.path.isdir(case_path):
        # Print the case folder name and the files/folders inside it
        print(name, "->", os.listdir(case_path))



validation_01 -> ['attribution_ACRIN-FLT-Breast.md', 'mask', 'PT']
validation_02 -> ['attribution_ACRIN-FLT-Breast.md', 'mask', 'PT']
validation_03 -> ['attribution_ACRIN-NSCLC-FDG-PET.md', 'mask', 'PT']
validation_04 -> ['attribution_ACRIN-NSCLC-FDG-PET.md', 'mask', 'PT']
validation_05 -> ['attribution_ACRIN-NSCLC-FDG-PET.md', 'mask', 'PT']
validation_06 -> ['attribution_ACRIN-NSCLC-FDG-PET.md', 'PT', 'mask']
validation_07 -> ['attribution_ACRIN-NSCLC-FDG-PET.md', 'PT', 'mask']
validation_08 -> ['attribution_ACRIN-NSCLC-FDG-PET.md', 'PT', 'mask']
validation_09 -> ['attribution_ACRIN-NSCLC-FDG-PET.md', 'mask', 'PT']
validation_10 -> ['attribution_ACRIN-NSCLC-FDG-PET.md', 'mask', 'PT']
validation_11 -> ['attribution_ACRIN-NSCLC-FDG-PET.md', 'mask', 'PT']
validation_12 -> ['attribution_ACRIN-NSCLC-FDG-PET.md', 'PT', 'mask']
validation_13 -> ['attribution_QIN-BREAST.md', 'mask', 'PT']
validation_14 -> ['attribution_NSCLC-Radiogenomics.md', 'PT', 'mask']
validation_15 -> ['attribution_Soft

In [16]:
# Pick the first case folder found in ROOT_DIR
case_dir = find_case_dirs(ROOT_DIR)[0]

# Build the mask folder path
mask_dir = os.path.join(case_dir, "mask")

# Find the first NIfTI mask file
mask_path = find_mask_file(mask_dir)

# Load the NIfTI file
nii = nib.load(mask_path)

# Print file name, shape, and affine matrix
print("Mask file:", mask_path)
print("Mask shape:", nii.shape)
print("Affine:\n", nii.affine)



Mask file: /content/drive/MyDrive/SUV-IBSI/Task 3_suv_computation-main/Real Patients/validation_01/mask/mask.nii.gz
Mask shape: (144, 144, 45)
Affine:
 [[-4.00000000e+00  0.00000000e+00  0.00000000e+00  2.86585938e+02]
 [ 0.00000000e+00 -4.00000000e+00  0.00000000e+00  2.16585938e+02]
 [ 0.00000000e+00  0.00000000e+00  4.00000000e+00  1.18650000e+03]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]


In [17]:
# Run the full SUV summary pipeline across all cases
df = run_all_cases(
    ROOT_DIR,          # folder containing all case folders
    OUTPUT_CSV,        # where to save the CSV summary
    erode_voxels=0     # use the exact reference ROI without extra processing
)

# Display the result table
df



validation_01: Units=BQML, DecayCorrection=START, Nvox=515, SUVmin=3.93, SUVmedian=4.98, SUVmax=5.64
validation_02: Units=BQML, DecayCorrection=START, Nvox=515, SUVmin=4.14, SUVmedian=5.84, SUVmax=7.75


/tmp/ipykernel_571/545158572.py:449: UserWarning: Only administration time available; reference date is assumed
  warnings.warn("Only administration time available; reference date is assumed")


validation_03: Units=BQML, DecayCorrection=START, Nvox=521, SUVmin=1.13, SUVmedian=1.78, SUVmax=2.86
validation_04: Units=GML, DecayCorrection=NONE, Nvox=525, SUVmin=2.36, SUVmedian=2.79, SUVmax=3.34
validation_05: Units=BQML, DecayCorrection=START, Nvox=521, SUVmin=1.21, SUVmedian=1.83, SUVmax=3.03
validation_06: ERROR -> Missing or zero PatientWeight
validation_07: Units=BQML, DecayCorrection=START, Nvox=481, SUVmin=1.51, SUVmedian=2.51, SUVmax=4.10
validation_08: ERROR -> Missing or zero PatientWeight
validation_09: Units=CNTS, DecayCorrection=START, Nvox=515, SUVmin=1.26, SUVmedian=1.54, SUVmax=1.84
validation_10: Units=BQML, DecayCorrection=START, Nvox=521, SUVmin=1.30, SUVmedian=2.09, SUVmax=3.26
validation_11: ERROR -> Missing or invalid ActualFrameDuration
validation_12: ERROR -> Missing or invalid RadionuclideTotalDose


/tmp/ipykernel_571/545158572.py:442: UserWarning: Administration date is inconsistent; using reference date and administration time
  warnings.warn("Administration date is inconsistent; using reference date and administration time")


validation_13: Units=BQML, DecayCorrection=START, Nvox=353, SUVmin=1.41, SUVmedian=1.95, SUVmax=2.48
validation_14: ERROR -> CNTS without valid Philips scale factors
validation_15: Units=BQML, DecayCorrection=START, Nvox=653, SUVmin=0.44, SUVmedian=0.79, SUVmax=1.94
validation_16: Units=BQML, DecayCorrection=START, Nvox=657, SUVmin=4.32, SUVmedian=5.75, SUVmax=7.40

FRESH RESULTS CSV: /content/drive/MyDrive/SUV-IBSI/Task 3_suv_computation-main/Results/Result 5_run_20260927_212904_503332.csv
Verified 16 result rows; mask erosion = 0.
Previous fixed-name CSV remains unchanged: /content/drive/MyDrive/SUV-IBSI/Task 3_suv_computation-main/Results/Result 5.csv


,PatientDir,PatientName,PatientID,SUVmin,SUVmedian,SUVmax,SUVmean
0,validation_01,ACRIN-FLT-Breast_040,ACRIN-FLT-Breast_040,3.930,4.980,5.636,4.952
1,validation_02,ACRIN-FLT-Breast_049,ACRIN-FLT-Breast_049,4.140,5.840,7.746,5.854
2,validation_03,ACRIN-NSCLC-FDG-PET-003,ACRIN-NSCLC-FDG-PET-003,1.130,1.780,2.863,1.787
3,validation_04,ACRIN-NSCLC-FDG-PET-020,ACRIN-NSCLC-FDG-PET-020,2.360,2.790,3.337,2.803
4,validation_05,ACRIN-NSCLC-FDG-PET-049,ACRIN-NSCLC-FDG-PET-049,1.210,1.830,3.029,1.875
5,validation_06,ACRIN-NSCLC-FDG-PET-090,ACRIN-NSCLC-FDG-PET-090,ERROR,ERROR,ERROR,ERROR
6,validation_07,ACRIN-NSCLC-FDG-PET-108,ACRIN-NSCLC-FDG-PET-108,1.510,2.510,4.101,2.542
7,validation_08,ACRIN-NSCLC-FDG-PET-144,ACRIN-NSCLC-FDG-PET-144,ERROR,ERROR,ERROR,ERROR
8,validation_09,ACRIN-NSCLC-FDG-PET-157,ACRIN-NSCLC-FDG-PET-157,1.260,1.540,1.835,1.534
9,validation_10,ACRIN-NSCLC-FDG-PET-195,ACRIN-NSCLC-FDG-PET-195,1.300,2.090,3.261,2.104


In [20]:
# Create the output folder for SUV-corrected PET DICOM series if it does not exist
os.makedirs(SUV_OUTPUT_DIR, exist_ok=True)

# Prepare a list to store per-case save summaries
saved_cases = []

# Loop through all case folders
for case_dir in find_case_dirs(ROOT_DIR):
    # Extract just the case folder name
    case_name = os.path.basename(case_dir)

    try:
        # Convert PET to SUVbw and save it as a new DICOM series
        name, out_dir, shape, slope = save_suv_dicom_series(case_dir, SUV_OUTPUT_DIR)

        # Save summary info for this case
        saved_cases.append({
            "case": name,
            "output_folder": out_dir,
            "shape": str(shape),
            "rescale_slope": slope
        })

        # Print success message
        print(f"{name}: SAVED -> {out_dir} | shape={shape} | RescaleSlope={slope:.8f}")

    except Exception as e:
        # Print failure message and continue with the next case
        print(f"{case_name}: FAILED -> {e}")

# Convert summary list to a DataFrame
saved_df = pd.DataFrame(saved_cases).sort_values("case").reset_index(drop=True)

# Display saved-case summary table
saved_df


validation_01: SAVED -> /content/drive/MyDrive/SUV-IBSI/Task 3_suv_computation-main/Corrected Images/validation_01 | shape=(45, 144, 144) | RescaleSlope=0.00073086


validation_02: SAVED -> /content/drive/MyDrive/SUV-IBSI/Task 3_suv_computation-main/Corrected Images/validation_02 | shape=(255, 144, 144) | RescaleSlope=0.00413896


/tmp/ipykernel_571/545158572.py:449: UserWarning: Only administration time available; reference date is assumed
  warnings.warn("Only administration time available; reference date is assumed")


validation_03: SAVED -> /content/drive/MyDrive/SUV-IBSI/Task 3_suv_computation-main/Corrected Images/validation_03 | shape=(205, 128, 128) | RescaleSlope=0.00066416


validation_04: SAVED -> /content/drive/MyDrive/SUV-IBSI/Task 3_suv_computation-main/Corrected Images/validation_04 | shape=(172, 144, 144) | RescaleSlope=0.00045434


validation_05: SAVED -> /content/drive/MyDrive/SUV-IBSI/Task 3_suv_computation-main/Corrected Images/validation_05 | shape=(239, 128, 128) | RescaleSlope=0.00109045
validation_06: FAILED -> Missing or zero PatientWeight


... (4 outputs left out)

validation_13: SAVED -> /content/drive/MyDrive/SUV-IBSI/Task 3_suv_computation-main/Corrected Images/validation_13 | shape=(83, 128, 128) | RescaleSlope=0.00017620
validation_14: FAILED -> CNTS without valid Philips scale factors


validation_15: SAVED -> /content/drive/MyDrive/SUV-IBSI/Task 3_suv_computation-main/Corrected Images/validation_15 | shape=(267, 128, 128) | RescaleSlope=0.00027879


validation_16: SAVED -> /content/drive/MyDrive/SUV-IBSI/Task 3_suv_computation-main/Corrected Images/validation_16 | shape=(324, 168, 168) | RescaleSlope=0.00024529


,case,output_folder,shape,rescale_slope
0,validation_01,/content/drive/MyDrive/SUV-IBSI/Task 3_suv_com...,"(45, 144, 144)",0.000731
1,validation_02,/content/drive/MyDrive/SUV-IBSI/Task 3_suv_com...,"(255, 144, 144)",0.004139
2,validation_03,/content/drive/MyDrive/SUV-IBSI/Task 3_suv_com...,"(205, 128, 128)",0.000664
3,validation_04,/content/drive/MyDrive/SUV-IBSI/Task 3_suv_com...,"(172, 144, 144)",0.000454
4,validation_05,/content/drive/MyDrive/SUV-IBSI/Task 3_suv_com...,"(239, 128, 128)",0.001090
5,validation_07,/content/drive/MyDrive/SUV-IBSI/Task 3_suv_com...,"(246, 128, 128)",0.001365
6,validation_09,/content/drive/MyDrive/SUV-IBSI/Task 3_suv_com...,"(213, 144, 144)",0.001052
7,validation_10,/content/drive/MyDrive/SUV-IBSI/Task 3_suv_com...,"(195, 128, 128)",0.000265
8,validation_13,/content/drive/MyDrive/SUV-IBSI/Task 3_suv_com...,"(83, 128, 128)",0.000176
9,validation_15,/content/drive/MyDrive/SUV-IBSI/Task 3_suv_com...,"(267, 128, 128)",0.000279
